# 04 — Ablation Study

최종 개선 모델의 성능 향상이 어떤 요소에서 왔는지 하나씩 확인하는 노트북입니다.

누적 ablation 순서:

- A0 = M1 Item Query 기본형
- A1 = A0 + Differential LR
- A2 = A1 + Multi-order Oversampling
- A3 = A2 + Weighted Loss = M2
- A4 = A3 + R-Drop (선택)

처음에는 seed 42 하나로 screening하고, 논문에 ablation 표를 넣을 때만 42/43/44 세 seed로 확장하세요.


## 1. 공개 저장소 clone 및 패키지 설치

In [ ]:
import os, subprocess
from pathlib import Path

repo_dir = Path('/content/pumpkin_public')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/hammonsterwow/pumpkin_public.git',
        str(repo_dir),
    ], check=True)

os.chdir(repo_dir)
subprocess.run([
    'pip', 'install', '-q', '-r',
    'experiments/nlu/kips_2026_item_query/requirements-colab.txt',
], check=True)
print('repo:', repo_dir)


## 2. GPU 및 ablation 설정 확인

In [ ]:
import torch
from experiments.nlu.kips_2026_item_query.config import ABLATION_CONFIGS, RDROP_CONFIG, MODEL_NAME
print('ablation configs:')
for c in ABLATION_CONFIGS: print(' -', c)
print('optional:', RDROP_CONFIG)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


## 3. 동일 데이터 split 준비

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from experiments.nlu.kips_2026_item_query.run_all import canonical_paths, prepare_data, save_manifest
train_valid_path, test_path = canonical_paths()
pool, train_records, valid_records, test_records = prepare_data(train_valid_path, test_path)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
result_root = Path('/content/kips_ablation_results')
result_root.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(result_root / 'tokenizer')
save_manifest(result_root, train_valid_path, test_path, pool, train_records, valid_records, test_records)


## 4. 1차 Ablation screening — seed 42

A0→A1→A2→A3를 같은 데이터/seed로 비교합니다. A0는 M1, A3는 M2와 동일 설정입니다. 새 Colab 세션에서는 결과 캐시가 없으므로 다시 학습합니다.


In [ ]:
from experiments.nlu.kips_2026_item_query.train_eval import run_suite, summarize
ablation_runs_42 = run_suite(ABLATION_CONFIGS, [42], train_records, valid_records, test_records, tokenizer, result_root)
ablation_runs_42.to_csv(result_root / 'ablation_runs_seed42.csv', index=False)
ablation_summary_42 = summarize(ablation_runs_42)
ablation_summary_42.to_csv(result_root / 'ablation_summary_seed42.csv', index=False)
display(ablation_summary_42[['experiment','multi_item_em_mean','order_exact_match_mean','frame_accuracy_mean']])


## 5. 선택: R-Drop screening — seed 42

R-Drop은 계산량이 커지므로 메인 실험과 A0~A3 결과를 먼저 본 뒤 실행하세요.


In [ ]:
RUN_RDROP = False  # 필요할 때만 True
if RUN_RDROP:
    rdrop_runs = run_suite([RDROP_CONFIG], [42], train_records, valid_records, test_records, tokenizer, result_root)
    rdrop_runs.to_csv(result_root / 'rdrop_seed42.csv', index=False)
    display(rdrop_runs)


## 6. 선택: Ablation 전체를 3 seeds로 재실행

seed 42 screening에서 차이가 의미 있어 논문에 ablation 표를 넣기로 했을 때만 실행하세요. 이미 완료된 seed 42 run은 `result.json` cache를 재사용합니다.


In [ ]:
RUN_THREE_SEEDS = False  # 논문에 ablation을 넣기로 했을 때 True
if RUN_THREE_SEEDS:
    ablation_runs_all = run_suite(ABLATION_CONFIGS, [42,43,44], train_records, valid_records, test_records, tokenizer, result_root)
    ablation_runs_all.to_csv(result_root / 'ablation_runs_3seeds.csv', index=False)
    ablation_summary_all = summarize(ablation_runs_all)
    ablation_summary_all.to_csv(result_root / 'ablation_summary_3seeds.csv', index=False)
    display(ablation_summary_all[['experiment','n_seeds','multi_item_em_mean','multi_item_em_std','order_exact_match_mean','order_exact_match_std']])


## 7. 결과 ZIP 다운로드

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/kips_ablation_results', 'zip', result_root)
files.download(zip_path)
